# RQ3 Charts
**Era Trend Lines & 5-Model Forecast Comparison**

Reads from `data/feature/movies_features.csv` and `output/model_metrics.csv`.

In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score

PROJECT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT, "src", "algorithms"))
from config import FEAT_DATA_PATH, OUTPUT_DIR, OUTPUT_CHART_DIR, F1_COLS, EMOTION_NAMES, ERAS

df = pd.read_csv(FEAT_DATA_PATH)
metrics_df = pd.read_csv(os.path.join(OUTPUT_DIR, "model_metrics.csv"))
print(f"Loaded {len(df)} films")
print(metrics_df[["model_name","r2","rmse"]])

Loaded 1466 films
           model_name      r2    rmse
0  Polynomial (deg-2)  0.9433  1.4804
1       Random Forest  0.9191  1.7689
2   Linear Regression  0.3089  5.1689
3           SVR (RBF)  0.3046  5.1849
4    Ridge Regression  0.2664  5.3253


## Chart 1 — Era Trend Lines (Top 8 Volatile Emotions)

In [2]:
era_order = list(ERAS.keys())
era_means = df.groupby("era")[F1_COLS].mean()
era_means = era_means.reindex([e for e in era_order if e in era_means.index])
era_means.columns = EMOTION_NAMES

year_means = df.groupby("year")[F1_COLS].mean()
year_means.columns = EMOTION_NAMES
top8 = year_means.std().sort_values(ascending=False).head(8).index.tolist()

fig, ax = plt.subplots(figsize=(14, 8))
x_pos = range(len(era_means))
colors = sns.color_palette("tab10", len(top8))
for i, emo in enumerate(top8):
    if emo in era_means.columns:
        ax.plot(x_pos, era_means[emo].values, marker="o", label=emo, color=colors[i], linewidth=2)
ax.set_xticks(list(x_pos))
ax.set_xticklabels(list(era_means.index), fontsize=9)
ax.set_xlabel("Cinema Era", fontsize=12)
ax.set_ylabel("Average Emotion Score (f1)", fontsize=12)
ax.set_title("RQ3: Emotion Shifts Across 4 Cinema Eras", fontsize=14)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(OUTPUT_CHART_DIR, "rq3_era_trend_line.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

Saved: C:\Users\siucomp\Desktop\Review 2\output\charts\rq3_era_trend_line.png


## Chart 2 — 5-Model Forecast Comparison

In [3]:
EMOTIONS_TO_FORECAST = ["nostalgia", "catharsis", "anger"]
future_years = np.arange(2025, 2036)

models_dict = {
    "Linear Regression":  LinearRegression(),
    "Ridge Regression":   Ridge(alpha=1.0),
    "Polynomial (deg-2)": make_pipeline(PolynomialFeatures(degree=2), LinearRegression()),
    "Random Forest":      RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR (RBF)":          SVR(kernel="rbf", C=1.0, epsilon=0.1),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, emo in zip(axes, EMOTIONS_TO_FORECAST):
    if emo not in year_means.columns:
        continue
    series = year_means[emo].dropna()
    emo_years = np.array(series.index)
    values = series.values
    all_years = np.concatenate([emo_years, future_years])

    ax.scatter(emo_years, values, alpha=0.4, s=15, color="gray", label="Original data")
    colors5 = sns.color_palette("muted", 5)
    best_r2, best_name, best_model = -999, "", None

    for i, (name, model) in enumerate(models_dict.items()):
        model.fit(emo_years.reshape(-1, 1), values)
        y_all = model.predict(all_years.reshape(-1, 1))
        r2 = r2_score(values, model.predict(emo_years.reshape(-1, 1)))
        ax.plot(all_years, y_all, alpha=0.35, linewidth=1, color=colors5[i], linestyle="--")
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model

    y_best = best_model.predict(all_years.reshape(-1, 1))
    ax.plot(all_years, y_best, linewidth=2.5, color="crimson",
            label=f"Best: {best_name}")
    ax.axvline(2024.5, color="black", linestyle=":", linewidth=1.2, label="Year 2024")
    ax.set_title(f"Emotion: {emo.capitalize()}", fontsize=12)
    ax.set_xlabel("Year")
    ax.set_ylabel("Avg f1 Score")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle("RQ3: 5-Model Forecast of Emotion Trajectories (2025-2035)", fontsize=14)
plt.tight_layout()
out = os.path.join(OUTPUT_CHART_DIR, "rq3_5models_forecast.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

Saved: C:\Users\siucomp\Desktop\Review 2\output\charts\rq3_5models_forecast.png
